In [1]:
import torch
import torch.nn as nn
import numpy as np
from VIX_Heston_util import *

In [2]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on {device}')

In [3]:
sequence_length = 30
dt = 1/365
T = dt * sequence_length
K = 100
option_type = 'put'
transaction_cost_rate = 2e-3
transaction_cost_vix = 2e-3

In [4]:
def evaluate_performance(network, S, V, VIX, F1, p0):
    network.eval()
    with torch.no_grad():
        feats = construct_features(S, VIX, F1, K, T, dt, sequence_length)
        holding = network(feats)
        dS = holding[:, :, 0]
        dF1 = holding[:, :, 1]
        
        delta_S = S[:, 1:] - S[:, :-1]
        delta_F1 = F1[:, 1:] - F1[:, :-1]
        
        # Transaction costs
        dS_shifted = torch.cat((torch.zeros((S.shape[0], 1), device=device), dS[:, :-1]), dim=1)
        dF1_shifted = torch.cat((torch.zeros((F1.shape[0], 1), device=device), dF1[:, :-1]), dim=1)
        
        cost_S = transaction_cost_rate * S[:, :-1] * torch.abs(dS - dS_shifted)
        cost_F1 = transaction_cost_vix * F1[:, :-1] * 5.0 * torch.abs(dF1 - dF1_shifted) # 5.0 multiplier for VIX
        
        PnL_S = (dS * delta_S) - cost_S
        PnL_F1 = (dF1 * delta_F1) - cost_F1
        
        Total_PnL = PnL_S.sum(dim=1) + PnL_F1.sum(dim=1)
        
        if option_type == 'put':
            C_T = torch.max(K - S[:, -1], torch.tensor(0.0, device=device))
        else:
            C_T = torch.max(S[:, -1] - K, torch.tensor(0.0, device=device))
        
        # MSE Metric
        mse_error = torch.mean((C_T - Total_PnL - p0)**2).item()
        
        # Mean PnL Metric
        mean_pnl = torch.mean(Total_PnL).item()
        
        # CVaR (Expected Shortfall) Metric at 95%
        X = C_T - Total_PnL - p0
        alpha_cvar = 0.5
        sorted_X, _ = torch.sort(X)
        cvar_idx = int((1 - alpha_cvar) * len(sorted_X))
        cvar_error = torch.mean(sorted_X[cvar_idx:]).item()
        
    return {
        'MSE': mse_error,
        'Mean Realized PnL': mean_pnl,
        'CVaR_0.5': cvar_error
    }

In [5]:
# Load the data
data_path = '../Data/VIX_Heston_val.pt'
data = torch.load(data_path, map_location=device)
S, V, VIX, F1 = [t.to(device) for t in data]

In [6]:
# Load the network and p0
mdl_path = '../Result/VIX_HestonClean_mse_Put_N1e5_tran2e-3_part1.pth'
network = RNN_BN_simple_VIX(sequence_length=sequence_length, input_size=6, output_size=2, hidden_size=64, device=device)
network.load_state_dict(torch.load(mdl_path, map_location=device))
network.to(device)
network.eval()

# Optional: Load learned p0 if available, else set to theoretical BS or an estimate
p0_learned = 1.69 # Example placeholder, should be loaded from checkpoint if optimized

In [7]:
# Evaluate performance
results = evaluate_performance(network, S, V, VIX, F1, p0_learned)

print("Evaluation Metrics:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

In [8]:
import scipy.stats as si

def bs_put_delta(S, K, T, t, sigma, r=0.0):
    tau = T - t
    tau = np.maximum(tau, 1e-6)
    d1 = (np.log(S/K) + (r + 0.5 * sigma**2)*tau) / (sigma * np.sqrt(tau))
    return si.norm.cdf(d1) - 1.0

def bs_vega_hedge(S, K, T, t, sigma, r=0.0):
    tau = T - t
    tau = np.maximum(tau, 1e-6)
    d1 = (np.log(S/K) + (r + 0.5 * sigma**2)*tau) / (sigma * np.sqrt(tau))
    vega = S * np.sqrt(tau) * si.norm.pdf(d1)
    return vega / 100.0

def evaluate_bs_performance(S, VIX, F1, p0):
    S_np = S.cpu().numpy()
    VIX_np = VIX.cpu().numpy()
    
    seq_len = S_np.shape[1] - 1
    t_steps = np.arange(seq_len) * dt
    t_arr = np.tile(t_steps, (S_np.shape[0], 1))
    
    sigma_np = VIX_np[:, :-1] / 100.0
    S_path = S_np[:, :-1]
    
    dS_bs_np = bs_put_delta(S_path, K, T, t_arr, sigma_np)
    dF1_bs_np = bs_vega_hedge(S_path, K, T, t_arr, sigma_np)
    
    dS_bs = torch.tensor(dS_bs_np, dtype=torch.float32, device=device)
    dF1_bs = torch.tensor(dF1_bs_np, dtype=torch.float32, device=device)
    
    delta_S = S[:, 1:] - S[:, :-1]
    delta_F1 = F1[:, 1:] - F1[:, :-1]
    
    dS_shifted = torch.cat((torch.zeros((S.shape[0], 1), device=device), dS_bs[:, :-1]), dim=1)
    dF1_shifted = torch.cat((torch.zeros((F1.shape[0], 1), device=device), dF1_bs[:, :-1]), dim=1)
    
    cost_S = transaction_cost_rate * S[:, :-1] * torch.abs(dS_bs - dS_shifted)
    cost_F1 = transaction_cost_vix * F1[:, :-1] * 5.0 * torch.abs(dF1_bs - dF1_shifted)
    
    PnL_S = (dS_bs * delta_S) - cost_S
    PnL_F1 = (dF1_bs * delta_F1) - cost_F1
    
    Total_PnL = PnL_S.sum(dim=1) + PnL_F1.sum(dim=1)
    
    if option_type == 'put':
        C_T = torch.max(K - S[:, -1], torch.tensor(0.0, device=device))
    else:
        C_T = torch.max(S[:, -1] - K, torch.tensor(0.0, device=device))
        
    mse_error = torch.mean((C_T - Total_PnL - p0)**2).item()
    mean_pnl = torch.mean(Total_PnL).item()
    
    X = C_T - Total_PnL - p0
    alpha_cvar = 0.5
    sorted_X, _ = torch.sort(X)
    cvar_idx = int((1 - alpha_cvar) * len(sorted_X))
    cvar_error = torch.mean(sorted_X[cvar_idx:]).item()
    
    return {
        'MSE': mse_error,
        'Mean Realized PnL': mean_pnl,
        'CVaR_0.5': cvar_error
    }

bs_results = evaluate_bs_performance(S, VIX, F1, p0_learned)

print("\n--- BS Strategy vs Deep Hedging Agent ---")
for k in results.keys():
    print(f"{k}:")
    print(f"  Agent: {results[k]:.4f}")
    print(f"  BS:    {bs_results[k]:.4f}")